In [118]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import mne
from pydeconv.utils import *
from pydeconv.pydeconv_sims_v2 import *
from pydeconv import *
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error
%matplotlib qt 


# raw     = mne.io.read_raw_eeglab(data_path + "629959_analysis.set", preload=True)

# # Initialize the model

# rERP_model = PyDeconv(settings = settings , features = features, eeg = raw)
# X_design = rERP_model.create_matrix()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# --- Build a compound ERP kernel ---
sacc_kernel = CompoundKernel(name="saccade_response", sfreq=500)
sacc_kernel.add(peak_latency=0.1,  amplitude=0.10, width=0.03, shape='gaussian', label='P1', modifier=lambda sacc_amplitude: .5 * sacc_amplitude * sacc_amplitude)
sacc_kernel.add(peak_latency=0.17, amplitude=-0.07, width=0.04, shape='gaussian', label='N170')
sacc_kernel.add(peak_latency=0.25, amplitude=0.04, width=0.05, shape='hanning',  label='P2')
sacc_kernel.plot()

fix_kernel = CompoundKernel(name="fixation_response", sfreq=500)
fix_kernel.add(peak_latency=0.1,  amplitude=0.10, width=0.03, shape='gaussian', label='P1')
fix_kernel.add(peak_latency=0.17, amplitude=-0.07, width=0.04, shape='gaussian', label='N170')
fix_kernel.add(peak_latency=0.25, amplitude=0.04, width=0.05, shape='hanning',  label='P2')
fix_kernel.plot()

# --- Create random event onsets ---
sfreq = 500
duration = 6000  # seconds
n_events = 100
latencies = np.sort(np.random.choice(np.arange(500, int(duration * sfreq) - 500), n_events, replace=False))
events = pd.DataFrame({'latency': latencies, 'type': 'stimulus'})

# --- Simulate ---
# sim = EEGSimulator(sfreq=sfreq, duration=duration)
# sim.add_kernel('stimulus', kernel)
# sim.set_events(events)
# sim.simulate()
# sim.add_noise(colour='brown', scale=0.3)
# sim.plot()

{'_'}
{'_'}
{'_'}
{'_'}
{'_'}
{'_'}


In [122]:
# UnfoldSim notes
# unfoldsim first creates a table of *all* events on a trial. It is balanced in the sense that it's a cross product of all possible values
# You can however offset the events as you want. Then, to actually simulate the data you must place the events at a specific latency point.
# This is performed by onset_types. Each time you sample it you get the distance in samples from the next event. You can also conditionally change the distribution based on the dataframe.

# So, what we can do is:
# 1. Like we're doing, create a table of events on a trial. This is already done using TrialVariable and TrialStructure.
# 2. Then, we can have this distribution of event offsets, with conditional distribution. We can have a lambda that has access to the dataframe.
#   - Question: how do we control correlated variables? e.g. maybe we always want a fixation, saccade, fixation, sacade, ... pattern
#       - This is done by the generator function.
# 3. Then, we need to define the compound kernel like in the block above. Again, we might need access to the dataframe columns. If we have a few amount of kernels, we could generate each waveform for each table combination. Then, we convolve each one separately (this could be paralellized)


# Create sample TrialStructure
# Create a sample TrialStructure using the TrialVariable class

# Define trial variable
trial_result = TrialVariable(
    name='correct',
    generator=lambda n, _rng : np.random.choice([0, 1], size=n),
    static_accross_trial=True
)
trial_mss = TrialVariable(
    name='MSS',
    generator=lambda n, _rng : np.random.choice([1, 2, 3, 4], size=n),
    static_accross_trial=True
)

def alternate_fix_sacc(n, rng):
    if rng.choice([False, True]):
        base = ['fixation', 'saccade']
    else:
        base = ['saccade', 'fixation']

    return (base * int(((n + 1) / 2)))[:n]

stim_type = TrialVariable(
    name='stimulus_type',
    generator=lambda n, rng: alternate_fix_sacc(n, rng)
)
rt_var = TrialVariable(
    name='sacc_amplitude',
    generator=lambda n, rng: rng.uniform(low=0.0, high=1.0, size=n)
)

# Specify trial structure configuration
trial_vars = [trial_result, trial_mss, stim_type, rt_var]

# Initialize TrialStructure with a number of trials (e.g., 100)
trial_struct = TrialStructure(sfreq=sfreq, variables=trial_vars)

# Generate the events DataFrame
events_df = trial_struct.generate_events_df(n_events=n_events)

# Display the resulting DataFrame
print(events_df.head())

# Add latencies to the events dataframe
events_df = assign_event_latencies(events_df, sampler=build_uniform_isi_sampler(width=50, offset=20))

# Display the resulting DataFrame
print(events_df.head())


sim = EEGSimulator(sfreq=sfreq, duration=duration)
sim.add_kernel(lambda evt_df_row: evt_df_row[stim_type.name] == 'saccade', sacc_kernel)
sim.add_kernel(lambda evt_df_row: evt_df_row[stim_type.name] == 'fixation', fix_kernel)
sim.set_events(events_df)
sim.simulate()
sim.add_noise(colour='brown', scale=0.3)
sim.plot(event_category=lambda event_df_row: event_df_row[stim_type.name])

   correct  MSS stimulus_type  sacc_amplitude
0        1    4      fixation        0.738997
1        1    4       saccade        0.698484
2        1    4      fixation        0.399023
3        1    4       saccade        0.090568
4        1    4      fixation        0.811302
   correct  MSS stimulus_type  sacc_amplitude  latency
0        1    4      fixation        0.738997       43
1        1    4       saccade        0.698484       84
2        1    4      fixation        0.399023      123
3        1    4       saccade        0.090568      168
4        1    4      fixation        0.811302      217


TypeError: __autoreload_class__.<lambda>() missing 1 required positional argument: '_'